In [ ]:
from datetime import datetime
today = datetime.now().strftime("%Y-%m-%d")
print("Hello, world!")
print(f"Today's date prefix is: {today}")




In [1]:
from ramlytics_database import DynamoDBClient
db = DynamoDBClient()
import pandas as pd
import os

'''import dotenv
dotenv.load_dotenv()
user_id = os.getenv("CLERK_USER_ID")'''

class DbQuery:
    import dotenv
    dotenv.load_dotenv()
    #user_id = os.getenv("CLERK_USER_ID")

    def __init__(self, db_client, user_id=None):
        self.db = db_client
        if not user_id:
            user_id = os.getenv("CLERK_USER_ID")
        self.user_id = user_id
        self.set_instruments_df()
        self.set_accounts_df()

    def remove_dyndb_keycols(self, df, colnames=['PK', 'SK', 'GSI1PK', 'GSI1SK','GSI2PK', 'GSI2SK', 'GSI3PK', 'GSI3SK', 'GSI4PK', 'GSI4SK']):
        for col in df.columns:
            if col in colnames:
                df.drop(columns=[col], inplace=True)
        return df

    def set_instruments_df(self):
        self.instruments_df = self.remove_dyndb_keycols(pd.DataFrame(self.db.list_instruments()))
        return self.instruments_df

    def set_accounts_df(self):
        self.accounts_df = self.remove_dyndb_keycols(pd.DataFrame(self.db.list_accounts(self.user_id)))

        return self.accounts_df

    def get_account_positions(self, user_id=None):
        if not user_id:
            user_id = self.user_id or os.getenv("CLERK_USER_ID")
        
        accounts = self.db.list_accounts(user_id)

        if not accounts:
            raise ValueError("User has no accounts.")
        for account in accounts:
            account_id = account["account_id"]

            # loop through positions for the current account and collect them in a list
            positions_list = []
            for position in self.db.list_positions(account_id):
                symbol = position["symbol"]
                current_price = self.instruments_df[self.instruments_df["symbol"] == symbol]["current_price"].iloc[0]
                rec = {
                    "account_id": account_id,
                    "symbol": symbol,
                    "quantity": position["quantity"],
                    "avg_price_paid": position["holding"]["Price_Paid_Amt"],
                    "current_price": current_price, 
                }
                positions_list.append(rec)

            # convert positions_list to DataFrame and merge with positions_df
            if positions_list:
                positions_df = pd.merge(positions_df, pd.DataFrame(positions_list), how="outer") if 'positions_df' in locals() else pd.DataFrame(positions_list)    

            self.positions_df = positions_df

        return positions_df

    def summarize_accounts_by_symbol(self, df=None):
        if df is None:
            if not hasattr(self, 'positions_df'):
                self.get_account_positions()
            df = self.positions_df

        symbol_summary_df = df.groupby("symbol").agg(
            total_quantity=pd.NamedAgg(column="quantity", aggfunc="sum"),
            avg_price_paid=pd.NamedAgg(column="avg_price_paid", aggfunc="mean"),
            current_price=pd.NamedAgg(column="current_price", aggfunc="mean"),
            # list of accounts
            accounts=pd.NamedAgg(column="account_id", aggfunc=lambda x: list(set(x)))
        ).reset_index()
        symbol_summary_df['total_value'] = symbol_summary_df['total_quantity'].astype(float) * symbol_summary_df['current_price'].astype(float)
        return symbol_summary_df

    def get_unique_symbols(self):
        if not hasattr(self, 'positions_df'):
            self.get_account_positions()
        return self.positions_df['symbol'].unique() if 'symbol' in self.positions_df.columns else []
    
qry = DbQuery(db)

In [2]:
qry.accounts_df

,holdings,created_at,account_purpose,clerk_user_id,account_id,updated_at,account_name,cash_balance,header,instruments,id,record_type
0,"{'GLD': {'Quantity': 5, 'Account': 'RAM BRKG -...",2026-09-08T01:46:56.795585+00:00,,user_3DM8poEbvBf5VXyVG4AGF9bdnoe,RAM BRKG -3543,2026-09-08T01:46:56.795585+00:00,RAM BRKG -3543,353.81,{'RAM BRKG -3543': {'Total_Gain_Amt': 34991.05...,"[GLD, XLK]",RAM BRKG -3543,account
1,"{'FSTA': {'Price_Paid_Amt': 52.8983, 'Total_Ga...",2026-09-08T01:46:58.080964+00:00,,user_3DM8poEbvBf5VXyVG4AGF9bdnoe,RAM ROLL -1969,2026-09-08T01:46:58.080964+00:00,RAM ROLL -1969,1032.8,{'RAM ROLL -1969': {'Days_Gain_Unrealized_Amt'...,"[FSTA, GLD]",RAM ROLL -1969,account
2,"{'LVHI': {'Days_Range': '0.00 - 0.00', 'Price_...",2026-09-09T12:13:02.326794+00:00,,user_3DM8poEbvBf5VXyVG4AGF9bdnoe,RAM ROTH -2359,2026-09-09T12:13:02.326794+00:00,RAM ROTH -2359,277.14,{'RAM ROTH -2359': {'Net_Account_Value': 23320...,"[FSTA, GLD, HODL, IDHQ, IVW, LVHI, QQQ, RYCEY,...",RAM ROTH -2359,account
3,"{'GLD': {'Price_Paid_Amt': 399.3117, 'Quantity...",2026-09-08T01:47:00.531864+00:00,,user_3DM8poEbvBf5VXyVG4AGF9bdnoe,SFM ROLL -8322,2026-09-08T01:47:00.531864+00:00,SFM ROLL -8322,2187.42,{'SFM ROLL -8322': {'Total_Gain_Amt': 32487.43...,"[FSTA, GLD, IDHQ, IVW, JNJ, LVHI, QQQ, RYCEY, ...",SFM ROLL -8322,account
4,"{'GLD': {'Total_Gain_Amt': 117.55, 'Last_Price...",2026-09-08T01:47:02.005459+00:00,,user_3DM8poEbvBf5VXyVG4AGF9bdnoe,SFM ROTH -2236,2026-09-08T01:47:02.005459+00:00,SFM ROTH -2236,77.52,"{'SFM ROTH -2236': {'Total_Gain_Pct': 3.18, 'D...","[FSTA, GLD, LVHI, SLV, SPY, VYM]",SFM ROTH -2236,account


In [3]:
positions_df = qry.get_account_positions()
# filter positions for a specific symbol
symbol = "GLD"
positions_df = positions_df[positions_df['symbol'] == symbol]
positions_df


,account_id,symbol,quantity,avg_price_paid,current_price
0,RAM BRKG -3543,GLD,5,389.1,399.72
3,RAM ROLL -1969,GLD,5,345.476,399.72
5,RAM ROTH -2359,GLD,50,260.2822,399.72
16,SFM ROLL -8322,GLD,30,399.3117,399.72
27,SFM ROTH -2236,GLD,15,398.9333,399.72


In [4]:
qry.get_unique_symbols()

<StringArray>
[  'GLD',   'XLK',  'FSTA',  'HODL',  'IDHQ',   'IVW',  'LVHI',   'QQQ',
 'RYCEY',   'VYM',   'WMT',   'JNJ',   'SLV',   'SPY']
Length: 14, dtype: str

In [5]:
symbol = "GLD"

symbol,qry.db.get_instrument(symbol)['current_price']

('GLD', Decimal('399.72'))

In [11]:
summary_df = qry.summarize_accounts_by_symbol()
# sort summary by total value
summary_df = summary_df.sort_values(by='total_value', ascending=False)
summary_df = summary_df.reset_index(drop=True)
summary_df.to_csv('summary.csv', index=False)
summary_df

,symbol,total_quantity,avg_price_paid,current_price,accounts,total_value
0,XLK,410,50.3926,187.87,"[RAM BRKG -3543, RAM ROTH -2359]",77026.70
1,QQQ,85,341.28835,718.36,"[SFM ROLL -8322, RAM ROTH -2359]",61060.60
2,VYM,360,123.612233,163.52,"[SFM ROTH -2236, SFM ROLL -8322, RAM ROTH -2359]",58867.20
3,LVHI,1150,42.798633,42.45,"[SFM ROTH -2236, SFM ROLL -8322, RAM ROTH -2359]",48817.50
4,GLD,105,358.62064,399.72,"[SFM ROLL -8322, RAM ROTH -2359, RAM ROLL -196...",41970.60
5,IVW,250,83.56545,139.82,"[SFM ROLL -8322, RAM ROTH -2359]",34955.00
6,IDHQ,700,45.25185,44.78,"[SFM ROLL -8322, RAM ROTH -2359]",31346.00
7,FSTA,369,53.2459,52.83,"[SFM ROTH -2236, SFM ROLL -8322, RAM ROLL -196...",19494.27
8,JNJ,50,162.0,275.23,[SFM ROLL -8322],13761.50
9,SPY,17,471.2206,770.19,[SFM ROTH -2236],13093.23


In [7]:
account_id = "RAM ROTH -2359"
account = qry.db.get_account(clerk_user_id=os.getenv("CLERK_USER_ID"), account_id=account_id)
account.pop("holdings")
account.pop("instruments")
acct_header = account.pop("header")
account, acct_header[account_id]


({'created_at': '2026-09-09T12:13:02.326794+00:00',
  'account_purpose': '',
  'clerk_user_id': 'user_3DM8poEbvBf5VXyVG4AGF9bdnoe',
  'GSI1SK': 'ACCOUNT#2026-09-09T12:13:02.326794+00:00#RAM ROTH -2359',
  'account_id': 'RAM ROTH -2359',
  'account_name': 'RAM ROTH -2359',
  'updated_at': '2026-09-09T12:13:02.326794+00:00',
  'cash_balance': Decimal('277.14'),
  'GSI4SK': 'ACCOUNT',
  'GSI1PK': 'USER#user_3DM8poEbvBf5VXyVG4AGF9bdnoe',
  'GSI4PK': 'ID#RAM ROTH -2359',
  'PK': 'USER#user_3DM8poEbvBf5VXyVG4AGF9bdnoe',
  'id': 'RAM ROTH -2359',
  'record_type': 'account',
  'SK': 'ACCOUNT#RAM ROTH -2359'},
 {'Net_Account_Value': Decimal('233201.74'),
  'Days_Gain_Unrealized_Amt': Decimal('0.35'),
  'Total_Gain_Amt': Decimal('62664.34'),
  'Days_Gain_Unrealized_Pct': Decimal('0'),
  'Total_Gain_Pct': Decimal('36.81'),
  '2026_Contributions': Decimal('0'),
  'Cash_Purchasing_Power': Decimal('277.14')})

In [8]:
# merge account and header information
merged_account = {**account, **acct_header[account_id]}
merged_account

{'created_at': '2026-09-09T12:13:02.326794+00:00',
 'account_purpose': '',
 'clerk_user_id': 'user_3DM8poEbvBf5VXyVG4AGF9bdnoe',
 'GSI1SK': 'ACCOUNT#2026-09-09T12:13:02.326794+00:00#RAM ROTH -2359',
 'account_id': 'RAM ROTH -2359',
 'account_name': 'RAM ROTH -2359',
 'updated_at': '2026-09-09T12:13:02.326794+00:00',
 'cash_balance': Decimal('277.14'),
 'GSI4SK': 'ACCOUNT',
 'GSI1PK': 'USER#user_3DM8poEbvBf5VXyVG4AGF9bdnoe',
 'GSI4PK': 'ID#RAM ROTH -2359',
 'PK': 'USER#user_3DM8poEbvBf5VXyVG4AGF9bdnoe',
 'id': 'RAM ROTH -2359',
 'record_type': 'account',
 'SK': 'ACCOUNT#RAM ROTH -2359',
 'Net_Account_Value': Decimal('233201.74'),
 'Days_Gain_Unrealized_Amt': Decimal('0.35'),
 'Total_Gain_Amt': Decimal('62664.34'),
 'Days_Gain_Unrealized_Pct': Decimal('0'),
 'Total_Gain_Pct': Decimal('36.81'),
 '2026_Contributions': Decimal('0'),
 'Cash_Purchasing_Power': Decimal('277.14')}

In [9]:
merged_account_df = qry.remove_dyndb_keycols(pd.DataFrame([merged_account]))
merged_account_df

,created_at,account_purpose,clerk_user_id,account_id,account_name,updated_at,cash_balance,id,record_type,Net_Account_Value,Days_Gain_Unrealized_Amt,Total_Gain_Amt,Days_Gain_Unrealized_Pct,Total_Gain_Pct,2026_Contributions,Cash_Purchasing_Power
0,2026-09-09T12:13:02.326794+00:00,,user_3DM8poEbvBf5VXyVG4AGF9bdnoe,RAM ROTH -2359,RAM ROTH -2359,2026-09-09T12:13:02.326794+00:00,277.14,RAM ROTH -2359,account,233201.74,0.35,62664.34,0,36.81,0,277.14


In [10]:
merged_account_df.to_dict()

{'created_at': {0: '2026-09-09T12:13:02.326794+00:00'},
 'account_purpose': {0: ''},
 'clerk_user_id': {0: 'user_3DM8poEbvBf5VXyVG4AGF9bdnoe'},
 'account_id': {0: 'RAM ROTH -2359'},
 'account_name': {0: 'RAM ROTH -2359'},
 'updated_at': {0: '2026-09-09T12:13:02.326794+00:00'},
 'cash_balance': {0: Decimal('277.14')},
 'id': {0: 'RAM ROTH -2359'},
 'record_type': {0: 'account'},
 'Net_Account_Value': {0: Decimal('233201.74')},
 'Days_Gain_Unrealized_Amt': {0: Decimal('0.35')},
 'Total_Gain_Amt': {0: Decimal('62664.34')},
 'Days_Gain_Unrealized_Pct': {0: Decimal('0')},
 'Total_Gain_Pct': {0: Decimal('36.81')},
 '2026_Contributions': {0: Decimal('0')},
 'Cash_Purchasing_Power': {0: Decimal('277.14')}}

In [1]:
from ramlytics_yfinance import yf_search

symbol="HODL"
yf_search(symbol)


{'exchange': 'BTS',
 'shortname': 'VanEck Bitcoin Trust',
 'quoteType': 'ETF',
 'symbol': 'HODL',
 'index': 'quotes',
 'score': 10148500.0,
 'typeDisp': 'ETF',
 'longname': 'VanEck Bitcoin ETF',
 'exchDisp': 'BATS Trading',
 'isYahooFinance': True,
 'current_price': 21.53}

In [ ]:
# local test for dyndb_load_json module

import backend.dyndb_load_json.dyndb_load_json as dyndb_load_json

import os
outpath = os.getenv("OUTPATH", "/tmp/accounts") 

import boto3
s3_bucket = os.getenv("S3_BUCKET", "fh-danelfin-289755104220")
s3_ingest_folder = "inbound/"
s3_client = boto3.client("s3")

# for each .json in the s3 bucket under the key prefix
response = s3_client.list_objects_v2(Bucket=s3_bucket, Prefix=s3_ingest_folder)
for obj in response.get("Contents", []):
    key = obj["Key"]

    # process only files in root folder
    if "/" in key[len(s3_ingest_folder):]:
        continue

    if key.endswith(".json") :
        #print(f"Found JSON file: s3://{s3_bucket}/{key}")

        # emulate an s3 ObjectCreated event for the lambda handler
        event = {
            "Records": [
                {
                    "s3": {
                        "bucket": {"name": s3_bucket},
                        "object": {"key": key},
                    }
                }
            ]
        }
        dyndb_load_json.lambda_handler(event, None) # local test

print("Finished processing all JSON files.")



Next:  
1.  package and deploy dyndb_load_json as a lambda function
    -   why no .venv in the folder?

2.  does danelfin need repackaged?  






In [ ]:
danelfin_api_key = os.getenv("DANELFIN_API_KEY")

import backend.danelfin.danelfin as danelfin
client = danelfin.DanelfinClient(api_key=danelfin_api_key)

from datetime import datetime

# 1. Historical data for a ticker
pypl_history = client.ranking(ticker="XLK")
pypl_history
